# 🧪 Prompt Lab: Customer Support Ticket Triage

**Concept:** Prompt Engineering
**Task:** Given a raw customer support ticket, classify its **urgency**, **category**, and produce a **one-line summary** for the support queue.

This notebook runs the **same triage task** through five different prompting strategies and compares the results:

1. Zero-shot (no instructions, bare task)
2. Zero-shot + System Prompt (role & constraints)
3. Few-shot (examples included)
4. Chain-of-Thought (explicit step-by-step reasoning)
5. Structured Output (forced JSON schema)

The goal isn't to find "the best prompt" in the abstract — it's to see, on real inputs, how much prompt design changes reliability, format consistency, and quality of an agent-facing output.


## 1. Setup

In [ ]:
# pip install anthropic  (uncomment if needed)
# !pip install anthropic

import os
import json
from anthropic import Anthropic

# Set your API key as an environment variable before running:
#   export ANTHROPIC_API_KEY="your-key-here"
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

MODEL = "claude-sonnet-4-5"  # swap models here to compare later if you want

def ask(system, user, max_tokens=400):
    """Small helper to call the API with an optional system prompt."""
    kwargs = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": user}],
    }
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text


## 2. Sample Tickets

A handful of realistic support tickets to run through every prompt variant. Having a fixed, shared test set is what makes the comparison fair.


In [ ]:
tickets = [
    "Hi, I was charged twice for my subscription this month. This is the second time it's happened and I need this fixed ASAP or I'm cancelling.",
    "Hey team, just wondering if you have a dark mode planned for the mobile app? Not urgent, just curious :)",
    "URGENT: Our production API integration has been returning 500 errors for the last 20 minutes and our checkout flow is completely down.",
    "I forgot my password and the reset email isn't arriving. Tried 3 times now.",
    "Loving the new dashboard update! One small thing - the export button is slightly misaligned on Firefox.",
]

for i, t in enumerate(tickets, 1):
    print(f"[{i}] {t}\n")


## 3. Variant 1 — Zero-Shot (Bare Prompt)

No system prompt, no examples, no formatting instructions. This is the baseline every other variant should beat.


In [ ]:
def zero_shot(ticket):
    prompt = f"Triage this support ticket: {ticket}"
    return ask(system=None, user=prompt)

result = zero_shot(tickets[0])
print(result)


## 4. Variant 2 — Zero-Shot + System Prompt

Same task, but now we give the model a clear role, a fixed set of categories, and explicit output fields via a system prompt.


In [ ]:
SYSTEM_V2 = """You are a support ticket triage assistant for a SaaS company.
For every ticket, classify:
- urgency: one of [low, medium, high, critical]
- category: one of [billing, bug, feature_request, account_access, other]
- summary: a single sentence summary for the support queue

Be consistent and concise."""

def zero_shot_system(ticket):
    prompt = f"Ticket: {ticket}"
    return ask(system=SYSTEM_V2, user=prompt)

result = zero_shot_system(tickets[0])
print(result)


## 5. Variant 3 — Few-Shot

We show the model 2-3 worked examples of tickets and their correct triage before asking it to do a new one. Few-shot usually improves consistency of format and category boundaries the most.


In [ ]:
FEW_SHOT_EXAMPLES = """Example 1:
Ticket: "The app crashes every time I try to upload a photo larger than 5MB."
urgency: medium
category: bug
summary: App crashes on photo uploads over 5MB.

Example 2:
Ticket: "Can you add the ability to export reports as CSV? Would be really helpful."
urgency: low
category: feature_request
summary: Customer requests CSV export for reports.

Example 3:
Ticket: "I can't log in at all, and my team's entire workday depends on this tool. Please help immediately."
urgency: critical
category: account_access
summary: Customer completely locked out, business-critical impact.
"""

def few_shot(ticket):
    prompt = f"""{FEW_SHOT_EXAMPLES}
Now triage this new ticket in the same format:
Ticket: "{ticket}"
"""
    return ask(system=SYSTEM_V2, user=prompt)

result = few_shot(tickets[0])
print(result)


## 6. Variant 4 — Chain-of-Thought

We explicitly ask the model to reason about *why* a ticket falls into a category/urgency before giving the final answer. Useful for borderline or ambiguous tickets.


In [ ]:
def chain_of_thought(ticket):
    prompt = f"""Ticket: "{ticket}"

Think step by step:
1. What is the customer's core problem?
2. Does this affect revenue, security, or ability to use the product at all?
3. Based on that, what urgency and category apply?

Then give your final answer in this format:
urgency: ...
category: ...
summary: ...
"""
    return ask(system=SYSTEM_V2, user=prompt, max_tokens=500)

result = chain_of_thought(tickets[2])  # try it on the "urgent outage" ticket
print(result)


## 7. Variant 5 — Structured Output (Forced JSON)

For real agent pipelines, free text isn't good enough — downstream code needs to parse the result. Here we force a strict JSON schema and parse it immediately.


In [ ]:
SYSTEM_V5 = """You are a support ticket triage assistant.
Respond with ONLY valid JSON, no other text, no markdown fences, matching exactly this schema:
{
  "urgency": "low | medium | high | critical",
  "category": "billing | bug | feature_request | account_access | other",
  "summary": "one sentence summary"
}"""

def structured_output(ticket):
    prompt = f'Ticket: "{ticket}"'
    raw = ask(system=SYSTEM_V5, user=prompt)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"error": "invalid JSON returned", "raw": raw}

result = structured_output(tickets[0])
print(json.dumps(result, indent=2))


## 8. Run All Variants Across All Tickets

Now the real comparison: run every strategy against every ticket and collect the results.


In [ ]:
strategies = {
    "zero_shot": zero_shot,
    "zero_shot_system": zero_shot_system,
    "few_shot": few_shot,
    "chain_of_thought": chain_of_thought,
    "structured_output": structured_output,
}

results = {}
for name, fn in strategies.items():
    print(f"Running strategy: {name}")
    results[name] = [fn(t) for t in tickets]

print("Done.")


In [ ]:
# Save raw results to disk so they can be reviewed later without re-calling the API
with open("prompt_lab_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved to prompt_lab_results.json")


## 9. Side-by-Side Comparison

Print all five outputs for a single ticket together, so the formatting/quality differences are easy to eyeball.


In [ ]:
ticket_index = 0  # change this to inspect a different ticket

print(f"TICKET: {tickets[ticket_index]}\n")
for name in strategies:
    print(f"--- {name} ---")
    print(results[name][ticket_index])
    print()


## 10. Evaluation — What I Learned

_Fill this in after actually running the notebook and reading the outputs. Some prompts to think about:_

- **Format consistency:** Which variants gave a parseable, consistent format every time? Which ones drifted?
- **Correctness on the ambiguous ticket** (ticket 5 — feature-request-like praise with a minor bug tucked in): did any variant misclassify it?
- **Correctness on the critical ticket** (ticket 3 — the outage): did every variant catch the urgency correctly, or did some undersell it?
- **JSON reliability:** did `structured_output` ever fail to parse? What would you change in the prompt to make it more robust?
- **Cost/latency tradeoff:** few-shot and chain-of-thought use more input/output tokens than zero-shot — was the quality gain worth it for this task?

**My takeaways:**

1. _(write 2-3 sentences here)_
2. _(write 2-3 sentences here)_
3. _(write 2-3 sentences here)_
